# Универсальный EDA — DATATHON 2026

Официальные данные и задача пока не заданы. Заполните параметры вручную после получения задания.
Ноутбук ничего не скачивает, не генерирует датасеты и не изменяет исходный файл.
Для облачного запуска сначала проверьте разрешение организаторов на размещение данных в Colab.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

# Запуск из корня проекта или notebooks/. В Colab сначала перейдите в папку репозитория.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AGENTS.md').is_file()), None)
if ROOT is None:
    raise RuntimeError('Перейдите в папку репозитория перед запуском ноутбука.')
sys.path.insert(0, str(ROOT))
DATA_PATH = None  # Path к реальному CSV/XLSX/Parquet; обязательный параметр
TARGET = None  # Название цели, только если она определена заданием
READ_OPTIONS = {}  # Например encoding, sep, dtype или sheet_name
OUTPUT_DIR = ROOT / 'outputs' / 'eda' / uuid4().hex
MAX_COLUMNS = 20


In [ ]:
from src.data_loader import load_data
from src.quality import quality_report, save_quality_report
from src.eda import plot_eda
from IPython.display import display, Image
if DATA_PATH is None:
    raise ValueError('Укажите DATA_PATH после получения официальных данных.')
path = Path(DATA_PATH)
path = path if path.is_absolute() else ROOT / path
frame = load_data(path, **READ_OPTIONS)
if TARGET is not None and TARGET not in frame:
    raise ValueError(f'Цель отсутствует: {TARGET}')
display(frame.head())
print(f'Строк: {len(frame)}, столбцов: {frame.shape[1]}')


In [ ]:
report = quality_report(frame)
display(frame.dtypes.to_frame('dtype'))
display(frame.isna().sum().to_frame('missing'))
display(frame.describe(include='all'))
print('Дубликаты:', report['duplicate_rows'])
display(report['flags'])
print(report['limitations'])
save_quality_report(frame, OUTPUT_DIR / 'quality')


In [ ]:
figures = plot_eda(frame, OUTPUT_DIR / 'figures', target=TARGET, max_columns=MAX_COLUMNS)
for figure in figures:
    display(Image(filename=str(figure)))
print('Сохранено в:', OUTPUT_DIR)


## Выводы — заполнить по фактическим наблюдениям

- Источник/версия данных и смысл одной строки: **UNKNOWN**.
- Качество данных и подозрительные значения: **NOT REVIEWED**.
- Доступность признаков на момент прогноза, возможные утечки: **NOT REVIEWED**.
- Временная зависимость и группы объектов: **NOT REVIEWED**.
- Цель и официальная метрика (если предусмотрены): **UNKNOWN**.
- Следующие проверки и ограничения: **NOT REVIEWED**.

Числовые коды категорий не становятся количественными признаками автоматически: проверьте типы.
Корреляция Pearson показана только для непостоянных числовых столбцов, без причинных выводов.
Графики ограничены MAX_COLUMNS (плюс TARGET); частоты показывают top 20 категорий.
Изучение всего датасета может влиять на выбор признаков: для подтверждающей оценки заранее отделите holdout.
Перед коммитом очистите выводы ноутбука от исходных данных.
